In [ ]:
from typing import TypedDict
import traceback
import re
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
import textwrap
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv


In [ ]:
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')



In [ ]:
llm = ChatOpenAI(
    model=model,
    api_key=api_key,
)

In [ ]:
# ---- Shared State ----
class AgentState(TypedDict):
    request: str        # user-defined task/request
    plan: str
    code: str
    test_output: str    # "SUCCESS" or "ERROR"
    documentation: str
    error: str


# ---- Helper functions ----
def call_llm(prompt: str) -> str:
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

def clean_code(code: str) -> str:
    """
    Remove Markdown code block fences (``` or ```python) from LLM output.
    """
    code = re.sub(r"^```(?:python)?\s*", "", code, flags=re.MULTILINE)
    code = re.sub(r"```$", "", code, flags=re.MULTILINE)
    return code.strip()

# ---- Nodes ----
def plan_node(state: AgentState) -> AgentState:
    # Use the request already stored in the state
    request = state.get("request", "")
    state["plan"] = f"Task request: {request}"
    print("[Plan_Task] Plan created from request.")
    return state


def generate_code_node(state: AgentState) -> AgentState:
    prompt = f"Generate Python code for the following task:\n{state['plan']}\nReturn only working Python code."
    code = call_llm(prompt)
    state["code"] = clean_code(code)
    print("[Generate_Code] Code generated and cleaned.")
    return state

def test_code_node(state: AgentState) -> AgentState:
    try:
        env = {}
        exec(clean_code(state["code"]), env)

        # If user included a test snippet (# TEST:) in their request, execute it
        test_snippet = re.search(r"# TEST:(.*)", state["request"], re.DOTALL)
        if test_snippet:
            # Extract the test code and clean up indentation
            test_code = test_snippet.group(1).strip()
            
            # Remove common leading whitespace (dedent)
            test_code = textwrap.dedent(test_code)
            
            # Execute the cleaned test code
            exec(test_code, env)
        
        state["test_output"] = "SUCCESS"
        state["error"] = ""
        print(f"[Test_Code] SUCCESS")

    except Exception:
        state["test_output"] = "ERROR"
        state["error"] = traceback.format_exc()
        print(f"[Test_Code] Failed:\n{state['error']}")

    return state

def debug_code_node(state: AgentState) -> AgentState:
    prompt = f"""The following code failed with this error:\n{state['error']}\n
    Original code:\n{state['code']}\n
    Please fix the code so it passes the test and return only the corrected Python code."""
    code = call_llm(prompt)
    state["code"] = clean_code(code)
    state["error"] = ""
    print("[Debug_Code] Code fixed and cleaned by LLM.")
    return state

def document_node(state: AgentState) -> AgentState:
    prompt = f"Write concise Python documentation for the following code:\n{state['code']}\n"
    state["documentation"] = call_llm(prompt)
    print("[Document_Artifact] Documentation generated by LLM.")
    return state

# ---- Conditional Edges ----
def decide_after_test(state: AgentState) -> str:
    return "document" if state["test_output"] == "SUCCESS" else "debug"

def decide_after_debug(state: AgentState) -> str:
    return "test"



In [ ]:
# ---- Build Workflow ----
workflow = StateGraph(AgentState)
workflow.add_node("plan", plan_node)
workflow.add_node("generate", generate_code_node)
workflow.add_node("test", test_code_node)
workflow.add_node("debug", debug_code_node)
workflow.add_node("document", document_node)

workflow.set_entry_point("plan")
workflow.add_edge("plan", "generate")
workflow.add_edge("generate", "test")
workflow.add_conditional_edges("test", decide_after_test, {"document":"document","debug":"debug"})
workflow.add_conditional_edges("debug", decide_after_debug, {"test":"test"})
workflow.add_edge("document", END)

app = workflow.compile()

In [ ]:
if __name__ == "__main__":
    user_request = input("Enter your coding request (optional test snippet with # TEST:): ")
    state: AgentState = {
        "request": user_request,  # store request in state
        "plan": "",
        "code": "",
        "test_output": "",
        "documentation": "",
        "error": ""
    }

    # Now invoke workflow directly
    final_state = app.invoke(state)

    print("\n=== Workflow Complete ===")
    print("Plan:", final_state["plan"])
    print("Code:\n", final_state["code"])
    print("Documentation:\n", final_state["documentation"])

#Two examples:
#Create a function that finds the maximum number in a list. # TEST: assert find_max([1, 5, 3, 9, 2]) == 9
#Create a function that reverses a string. #TEST: reverse_string("hallomark") = "kramollah"
